# NB-03 — Bengali T1 attack corpus (negation inversion)

## Why this notebook exists

NB-00 selected the 25-utterance attack set with `text.str.contains("না", regex=False)` — a
**substring** test. In Bengali, `না` occurs inside many ordinary words:

| word | meaning | is it a negation? |
|---|---|---|
| না | not | **yes** — standalone post-verbal particle |
| নাম | name | no |
| নারী | woman | no |
| জানা | knowing | no |

So the advertised "15 utterances with না" were substring hits. Checked on 2026-09-25:
**0 of the 25 carry a standalone `না`.** The frozen T5 attack set cannot test RQ2 at all.

## What this notebook does

Mines a **separate, T1-specific attack set** from the held-out test split — utterances that
really do contain the standalone negation particle — and emits, for each one, **both** a T1 and
a T5 target.

That last part matters. Measuring T1 and T5 on *different* utterances confounds edit type with
utterance difficulty. Measuring both on the **same** utterances makes it a paired comparison,
which is far more powerful at small n and is strictly better than the design in the plan.

`bn-asr-splits` is **not** modified. This publishes a new dataset alongside it.

## Kaggle settings

| Setting | Value |
|---|---|
| Accelerator | **None (CPU)** — do not spend GPU quota here |
| Internet | On |
| Inputs | `bn-asr-splits` · `ckpt-xlsr-100h` (tokenizer only, for token counts) |
| Output | `bn-attack-targets` |


## 1 · Configuration

In [ ]:
# Target number of HUMAN-VALIDATED utterances in the final corpus. The notebook
# works incrementally: it keeps every pair already marked 'y' and tops up from
# the unjudged pool, so earlier validation work is never thrown away.
N_T1_TARGET = 45
# Cap on NEW pairs sent for judgement in one round. Measured accept rate on the
# 2026-09-25 round was 17/30: 80% for candidates with no stranded negative-polarity
# item, 33% for those with one -- so the sampler prefers NPI-free candidates and
# this many fresh pairs should yield roughly 0.75x that in usable ones.
N_FRESH_MAX = 40
DUR_LO, DUR_HI = 2.0, 8.0   # same window NB-00 used, so attack cost stays comparable
SEED        = 1337

NEG = "না"            # the standalone post-verbal negation particle
# Deliberately NOT included: "নি" (verbal suffix, fused -- no token to delete),
# "নেই" (existential negative -- deleting it leaves no verb, that is a
# substitution not a deletion). "নয়" is deletable but is a copula rather than a
# particle; it is COUNTED below for information and excluded from the corpus.
ALSO_COUNT  = ["নি", "নেই", "নয়"]

# Bengali sentences end with a danda, and it attaches to the preceding token. So
# "আমি যাব না।" tokenises as [...,"না।"] and a bare `NEG in text.split()` test
# MISSES it -- along with every other sentence-final negation, which is the
# canonical post-verbal position and the plan's headline example
# ("ওষুধটা দেবেন না।"). Strip punctuation before the membership test.
PUNCT = "।॥,;:?!\"'()[]{}—–…."


## 2 · Boilerplate

In [ ]:
import os, sys, json, re, glob, time, unicodedata
import numpy as np, pandas as pd

SEEDED = np.random.RandomState(SEED)
WORK = "/kaggle/working"
T0 = time.time()

PROV = {"notebook": "NB-03-attack-targets", "seed": SEED, "n_t1_target": N_T1_TARGET,
        "duration_window": [DUR_LO, DUR_HI], "negation_particle": NEG,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}

!pip install -q transformers==4.44.2 2>&1 | tail -1
from transformers import Wav2Vec2Processor
import transformers
PROV["transformers"] = transformers.__version__
print(f"transformers {transformers.__version__}")

def list_inputs():
    out = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))
INPUTS = list_inputs()

SPLITS = next((p for p in INPUTS if "splits" in os.path.basename(p).lower()), None)
assert SPLITS, f"Attach bn-asr-splits. Visible: {[os.path.basename(p) for p in INPUTS]}"

ZW = dict.fromkeys(map(ord, "\u200c\u200d"), None)
def norm_bn(s):
    """Identical to NB-00/02/05/06. NFC, strip ZWJ/ZWNJ, collapse whitespace."""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

print("splits:", SPLITS)


## 3 · Mine the test split for real negation

The gate here is `NEG in text.split()` — whitespace-delimited tokens, never a substring. The
printout quantifies exactly how badly the substring test overcounts, because that number belongs
in the thesis as a data-curation note.


In [ ]:
man = pd.read_csv(f"{SPLITS}/manifest_full.csv")
test = man[man.split == "test"].copy()
test["text"] = test["text"].map(norm_bn)
test = test[(test.duration >= DUR_LO) & (test.duration <= DUR_HI)]
test = test[test.text.str.len() > 0]
print(f"test split, {DUR_LO}-{DUR_HI}s: {len(test)} utterances")

def bare(t):  return [w.strip(PUNCT) for w in t.split()]
def has_neg(t): return NEG in bare(t)

substr    = test[test.text.str.contains(NEG, regex=False)]
naive     = test[test.text.map(lambda t: NEG in t.split())]   # the buggy test
tokens    = test[test.text.map(has_neg)]                      # punctuation-aware
substr_only = len(substr) - len(tokens)

print(f"\n  contains '{NEG}' as a SUBSTRING : {len(substr):>6}  ({100*len(substr)/len(test):.1f}%)")
print(f"  bare-token test (misses 'না।') : {len(naive):>6}  ({100*len(naive)/len(test):.1f}%)")
print(f"  contains '{NEG}' as a TOKEN     : {len(tokens):>6}  ({100*len(tokens)/len(test):.1f}%)"
      f"   <- punctuation-aware")
print(f"    of which sentence-final       : "
      f"{int(tokens.text.map(lambda t: bare(t)[-1] == NEG).sum()):>6}"
      f"   <- the canonical post-verbal case")
print(f"  substring-only (false positives): {substr_only:>6}  "
      f"({100*substr_only/max(1,len(substr)):.0f}% of substring hits)")
print("\n  This is why the NB-00 attack set had 0 usable T1 utterances.")

for f in ALSO_COUNT:
    n_tok = int(test.text.map(lambda t: f in bare(t)).sum())
    n_sub = int(test.text.str.contains(f, regex=False).sum())
    print(f"  (for information) '{f}': {n_tok} as token, {n_sub} as substring")

PROV["test_pool"] = int(len(test))
PROV["neg_substring"] = int(len(substr))
PROV["neg_token"] = int(len(tokens))
PROV["neg_token_naive"] = int(len(naive))
PROV["substring_false_positives"] = int(substr_only)

assert len(tokens) >= N_T1_TARGET, (
    f"only {len(tokens)} test utterances carry a standalone '{NEG}', need {N_T1_TARGET}. "
    f"Widen DUR_LO/DUR_HI, or draw from the dev split as well and say so in the paper.")
print(f"\n{len(tokens)} candidates available for {N_T1_TARGET} slots")


## 4 · Flag the cases a human must look at

Deleting `না` does not always produce a clean semantic inversion. These heuristics do not decide
anything — they route attention. Every surviving pair still needs a native speaker's sign-off in
section 7.


In [ ]:
def flagit(t):   # NOT "flags": DataFrame.flags is reserved by pandas
    b = bare(t); out = []
    if b[0] == NEG:                         out.append("sentence-initial")
    if b.count(NEG) > 1:                    out.append(f"x{b.count(NEG)}")
    if "?" in t:                            out.append("interrogative")
    if len(b) <= 3:                         out.append("very-short")
    # "X না Y" between two content words is the DISJUNCTIVE reading -- "X or Y",
    # not negation. Found in the 2026-09-25 draft: "তিনি মধ্যমা না কনিষ্ঠা রাণী
    # ছিলেন" = "was she the middle OR the youngest queen", where deleting না
    # yields nonsense. Cannot be settled without syntax, so flag the position and
    # let the reviewer judge.
    if b[-1] != NEG:                        out.append("mid-sentence")
    return ";".join(out)

def neg_position(t):
    b = bare(t)
    return "final" if b[-1] == NEG else ("initial" if b[0] == NEG else "mid")

# Bengali negative concord: "না" licenses negative-polarity items elsewhere in
# the clause. Delete the particle and the NPI is stranded in a positive clause,
# which usually inverts the meaning correctly but often reads unidiomatically.
# Measured on the 2026-09-25 validation round: 80% accepted without an NPI,
# 33% with one. NOT a hard filter -- "কোন" appeared in 4 acceptances and 4
# rejections, so the call is contextual and stays with the human reviewer.
NPI = ["কোনো", "কোনও", "কোন", "কারও", "কাহারও", "কিছুই", "কখনও", "কখনো",
       "এক পয়সাও", "একটিও"]
def npi_in(t): return ",".join([w for w in NPI if w in t])

cand = tokens.copy()
cand["review_flags"] = cand.text.map(flagit)
cand["npi"] = cand.text.map(npi_in)
print("flagged candidates:")
print(cand.review_flags.value_counts().to_string())

# ---- incremental: keep what is already validated, top up the shortfall ----
VAL = next((f"{p}/t1_pairs_validated.csv" for p in INPUTS
            if os.path.exists(f"{p}/t1_pairs_validated.csv")), None)
judged, keep_y = {}, []
if VAL:
    _v = pd.read_csv(VAL)
    _v["validated"] = _v.validated.astype(str).str.strip().str.lower()
    judged = dict(zip(_v.source_utterance_id, _v.validated))
    # Carry the hand-assigned domain labels through. Without this they are
    # silently replaced by "" in the published corpus and the per-domain
    # analysis cannot be run from the dataset.
    DOMAIN = (dict(zip(_v.source_utterance_id, _v.domain.fillna("")))
              if "domain" in _v.columns else {})
    keep_y = [u for u, verdict in judged.items() if verdict == "y"]
    print(f"\nvalidated file: {len(judged)} judged, {len(keep_y)} accepted "
          f"({100*len(keep_y)/max(1,len(judged)):.0f}%)")
else:
    DOMAIN = {}
    print("\nno t1_pairs_validated.csv attached -- first round")

need = max(0, N_T1_TARGET - len(keep_y))
fresh_n = min(int(need / 0.75) + 1, N_FRESH_MAX, len(cand) - len(judged)) if need else 0
print(f"  have {len(keep_y)} validated, target {N_T1_TARGET} -> "
      f"sending {fresh_n} fresh pairs for judgement")

unjudged = cand[~cand.utt_id.isin(judged)]
# prefer unflagged AND NPI-free: that subgroup accepted at 80%
tier1 = unjudged[(unjudged.review_flags == "") & (unjudged.npi == "")]
tier2 = unjudged[(unjudged.review_flags == "") & (unjudged.npi != "")]
tier3 = unjudged[unjudged.review_flags != ""]
print(f"  unjudged pool: {len(tier1)} clean+NPI-free, {len(tier2)} clean+NPI, "
      f"{len(tier3)} flagged")
parts, left = [], fresh_n
for t in (tier1, tier2, tier3):
    if left <= 0: break
    take = min(left, len(t))
    if take: parts.append(t.sample(take, random_state=SEED)); left -= take
fresh = pd.concat(parts) if parts else unjudged.head(0)

pick = pd.concat([cand[cand.utt_id.isin(keep_y)], fresh])
pick = pick.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
FRESH_IDS = set(fresh.utt_id)
print(f"selected {len(pick)} = {len(keep_y)} validated + {len(fresh)} fresh")
PROV.update(n_judged=len(judged), n_accepted=len(keep_y), n_fresh=len(fresh))


## 5 · Build the T1 and T5 targets, on the SAME utterances

- **T1** removes the **last** standalone `না` — the post-verbal negation.
- **T5** is a length-matched sentence drawn from elsewhere in the test split, exactly as NB-05
  generates it, so the two edit types are directly comparable.

Because both targets attach to the same audio, ε50(T1) and ε50(T5) form a **paired** sample.


In [ ]:
CKPT = next((p for p in INPUTS if re.search(r"ckpt-xlsr-100h$", os.path.basename(p))), None)
assert CKPT, "Attach ckpt-xlsr-100h (tokenizer only -- no GPU needed)"
MDIR = CKPT if os.path.exists(f"{CKPT}/config.json") else f"{CKPT}/final"
proc = Wav2Vec2Processor.from_pretrained(MDIR)
def ntok(s): return len(proc.tokenizer(norm_bn(s)).input_ids)
print("tokenizer from:", MDIR)

def t1_target(t):
    """Delete the LAST standalone negation particle, carrying its punctuation
    back to the preceding word so "আমি যাব না।" -> "আমি যাব।" rather than
    losing the danda."""
    toks = t.split(); b = bare(t)
    idx = [k for k, w in enumerate(b) if w == NEG]
    if not idx: return None, -1
    j = idx[-1]
    trail = toks[j][len(toks[j].rstrip(PUNCT)):]
    out = toks[:j] + toks[j+1:]
    if trail and j > 0: out[j-1] = out[j-1].rstrip(PUNCT) + trail
    return norm_bn(" ".join(out)), j

def cer_chars(a, b):
    m, n = len(a), len(b); d = list(range(n+1))
    for i in range(1, m+1):
        pv = d[0]; d[0] = i
        for j in range(1, n+1):
            cur = d[j]; d[j] = min(d[j]+1, d[j-1]+1, pv + (a[i-1] != b[j-1])); pv = cur
    return d[n]

pool = test[~test.utt_id.isin(pick.utt_id)].copy()
pool["ntok"] = pool.text.map(ntok)

rows = []
for _, r in pick.iterrows():
    src = r["text"]
    t1, pos = t1_target(src)
    if not t1 or t1 == src: continue
    st, t1n = ntok(src), ntok(t1)
    # Match T5 to the T1 TARGET's length, not the source's. Matching to the
    # source left T1 about 3 tokens shorter than T5 and gave a length-matched
    # subset of only 4 utterances in the 2026-09-25 draft, which is the control
    # the plan calls mandatory.
    c = pool[pool.ntok.between(t1n - 2, t1n + 2)]
    if len(c) == 0: continue
    t5 = norm_bn(c.sample(1, random_state=int(SEEDED.randint(1e6))).iloc[0]["text"])
    for et, tgt, lbl in (("T1", t1, "inversion"), ("T5", t5, "unrelated")):
        rows.append(dict(
            pair_id=f"{r['utt_id']}_{et}", source_utterance_id=r["utt_id"],
            source_text=src, target_text=tgt, edit_type=et,
            edit_distance_chars=cer_chars(src, tgt),
            semantic_delta_label=lbl, domain=DOMAIN.get(r["utt_id"], ""), notes="",
            neg_token_index=pos if et == "T1" else -1,
            source_tokens=st, target_tokens=ntok(tgt),
            duration=r["duration"], path=r["path"],
            review_flags=r["review_flags"],
            neg_position=neg_position(src), npi=r["npi"],
            validated=judged.get(r["utt_id"], "")))
tg = pd.DataFrame(rows)
print(f"{len(tg)} rows = {tg.source_utterance_id.nunique()} utterances x 2 edit types")
assert tg.source_utterance_id.nunique() >= 12, (
    f"only {tg.source_utterance_id.nunique()} of {len(pick)} selected utterances "
    f"survived target construction -- check the length-matching pool")


## 6 · The length control

Attack cost rises with target token count regardless of anything linguistic. If T1 targets are
simply shorter than T5 targets, an ε50 difference is an artefact of brevity, not morphology.
The plan makes this control mandatory.

T1 is *by construction* one token shorter than its source. The point is to quantify that, report
ε50 normalised by target length alongside the raw figure, and identify the subset where T1 and T5
token counts coincide.


In [ ]:
piv = tg.pivot_table(index="source_utterance_id", columns="edit_type",
                     values="target_tokens")
print("target token counts")
print(f"  T1  mean {piv.T1.mean():.1f}  median {piv.T1.median():.0f}")
print(f"  T5  mean {piv.T5.mean():.1f}  median {piv.T5.median():.0f}")
print(f"  T1 - T5 mean difference: {(piv.T1 - piv.T5).mean():+.1f} tokens")

ed = tg.pivot_table(index="source_utterance_id", columns="edit_type",
                    values="edit_distance_chars")
print(f"\nedit distance from source (characters)")
print(f"  T1  mean {ed.T1.mean():.1f}   <- minimum-edit, the RQ2 claim")
print(f"  T5  mean {ed.T5.mean():.1f}")

matched = piv[(piv.T1 - piv.T5).abs() <= 1].index
print(f"\nlength-matched subset (|T1-T5| <= 1 token): {len(matched)} utterances")
tg["length_matched"] = tg.source_utterance_id.isin(matched)
PROV["length_matched_n"] = int(len(matched))
PROV["mean_edit_chars"] = {"T1": float(ed.T1.mean()), "T5": float(ed.T5.mean())}
if len(matched) < 10:
    print("  WARNING: too few for a length-matched analysis. Report normalised eps50 instead,")
    print("  and say the matched subset was underpowered.")


## 7 · Human validation — the step that cannot be automated

Everything above is mechanical. Whether deleting `না` genuinely inverts the meaning is a
judgement, and it is the one part of RQ2 that a native speaker has to own.

Fill the `validated` column in `t1_pairs_for_review.csv`:

| value | meaning |
|---|---|
| `y` | deleting `না` inverts the meaning, and the result is grammatical Bengali |
| `n` | it does not invert, or the result is ungrammatical — **drop this pair** |
| `?` | unsure — discuss with the second speaker |

### Four failure modes to watch for

**1. `না` as a disjunction, not a negation.** `X না Y` between two content words means
*"X or Y"*. Real example from the first draft:

> কিন্তু তিনি মধ্যমা **না** কনিষ্ঠা রাণী ছিলেন তা নিয়ে বিতর্ক আছে।
> *"There is debate over whether she was the middle **or** the youngest queen."*

Deleting `না` gives *"she was the middle youngest queen"* — nonsense, not an inversion.
**Mark these `n`.** The `mid-sentence` flag and `neg_position` column narrow where to look, but
only you can tell the two readings apart.

**2. The remainder is contradictory rather than inverted.** e.g.

> বিজ্ঞান অভিশাপ **না** হয়ে আশীর্বাদ হবে → বিজ্ঞান অভিশাপ হয়ে আশীর্বাদ হবে

*"science will be a blessing rather than a curse"* → *"science, being a curse, will be a
blessing"*. Self-contradictory, so not a clean inversion. Mark `n` or `?`.

**3. Negative concord left dangling.** This is the most common issue in the sentence-final set.
Bengali pairs `না` with negative-polarity items — `কোনো` / `কেউ` / `কারও` / `আর`
("any / anyone / no one / anymore"). Delete `না` and the partner is stranded in a positive clause:

> কেউ অন্ধ কুকুর তুলে নিবে **না**। → কেউ অন্ধ কুকুর তুলে নিবে।
> *"**No one** will pick up a blind dog"* → *"**Someone** will pick up a blind dog"*

The meaning **does** invert here, so this is usually `y`. But judge the remainder as Bengali, not
just as logic: `এখন আর তেমন চোখে পড়ে না।` → `এখন আর তেমন চোখে পড়ে।` leaves `আর` without its
negative partner and reads awkwardly. **Mark `?` when the inversion is correct but the result is
unidiomatic** — an attack target the model would never plausibly emit is a weak target.

**4. Another negation survives.** Deleting one `না` from a doubly-negated sentence leaves the
sentence still negative, so the meaning has not flipped:

> কোনো আত্মীয়তার সম্পর্ক **না** থাকাতে ... সংগত মনে করেন **নাই**।

The `x2` / `x3` flags catch the repeated-`না` cases; `নাই` and `নি` are different forms and are
not flagged. Check the remainder actually reads as the *opposite* of the source.

### Filling `domain`

| Label | Covers |
|---|---|
| `medical` | health, medicine, diagnosis, treatment, dosage |
| `financial` | money, amounts, accounts, transactions |
| `civic` | government, law, politics, elections, public services, rights |
| `religious` | theology, scripture, practice, religious identity or heritage |
| `general` | everything else — biography, history, nature, description, literature |

`religious` is an **addition to the taxonomy in §6.1 of the plan**, which listed only the first
three plus `general`. It is justified twice over: the mined corpus genuinely contains theological
and religious-heritage statements that `general` would mislabel, and in the Bengali-speaking
context a flipped religious assertion carries consequences at least as severe as a flipped medical
instruction. Adding a topic label does not touch any metric definition, so it does not affect
comparability with the co-researcher's arm.

Expect most rows to come out `general`. Bengali.AI Speech is crowdsourced **read** speech from
encyclopedic and literary sources, so it contains almost no medical instructions and no banking
commands — nobody recorded any. **If a domain ends up with fewer than about 5 utterances, drop the
per-domain breakdown** and report it as a limitation rather than force-fitting labels: a
domain-balanced target set would require recording new audio, which is future work.

Per the plan a **second** native speaker validates. Re-run this notebook with the completed CSV
attached and section 8 will filter to validated rows.


In [ ]:
# Fresh pairs go out for judgement; the published corpus keeps only validated ones.
rev = tg[(tg.edit_type == "T1") & tg.source_utterance_id.isin(FRESH_IDS)][
    ["pair_id", "source_utterance_id", "source_text", "target_text",
     "edit_distance_chars", "neg_position", "npi", "review_flags",
     "source_tokens", "target_tokens", "domain", "validated"]]

if len(rev):
    rev.to_csv(f"{WORK}/t1_pairs_for_review.csv", index=False)
    print(f"wrote t1_pairs_for_review.csv: {len(rev)} NEW pairs to judge")
    print("  fill 'validated' (y/n/?) and 'domain', then APPEND these rows to your")
    print("  existing t1_pairs_validated.csv -- do not replace it.")
    print("  The 'npi' column flags a stranded negative-polarity item: those accepted")
    print("  at 33% last round versus 80% without one, so read them closely.")
    pd.set_option("display.max_colwidth", 55)
    print()
    print(rev[["source_text", "target_text", "npi", "review_flags"]]
          .head(15).to_string(index=False))
else:
    print(f"no fresh pairs needed -- already at or above N_T1_TARGET={N_T1_TARGET}")

n_built = tg.source_utterance_id.nunique()
if not judged:
    print("\nFIRST ROUND: no validated file attached, so this publishes the UNVALIDATED")
    print("  draft to let a pilot attack run. Any result from it must be labelled")
    print("  auto-generated in the paper.")
    PROV["human_validated"] = False
    PROV["n_corpus"] = int(n_built)
else:
    tg = tg[tg.source_utterance_id.isin(keep_y)]
    n_ok = tg.source_utterance_id.nunique()
    print(f"\ncorpus: {n_ok} human-validated utterances (of {n_built} built this round)")
    PROV["human_validated"] = True
    PROV["n_corpus"] = int(n_ok)
    assert n_ok >= 12, (
        f"only {n_ok} validated utterances -- too few for any RQ2 analysis. Judge the "
        f"fresh pairs written above, append them to t1_pairs_validated.csv, re-run.")
    if n_ok < 25:
        print(f"  POWER WARNING: n={n_ok}. The paired T1-vs-T5 design compensates somewhat,")
        print(f"  but n<25 will be raised in review. Judge the fresh pairs and re-run to")
        print(f"  reach N_T1_TARGET={N_T1_TARGET} before the run that feeds the thesis.")


## 8 · Freeze and publish

Two files matter downstream:

- **`bn_attack_targets.csv`** — NB-05 already looks for this name and prefers it over
  auto-generation. Columns match what its merge expects.
- **`attack_eval_t1.csv`** — the T1 utterance list with audio paths, in the same shape as
  NB-00's `attack_eval_25.csv`. NB-05 reads this when `EDIT_TYPE == "T1"`.

`bn-asr-splits` is untouched.


In [ ]:
tg = tg.sort_values(["source_utterance_id", "edit_type"]).reset_index(drop=True)
tg.to_csv(f"{WORK}/bn_attack_targets.csv", index=False)

ev = (tg[tg.edit_type == "T1"][["source_utterance_id", "path", "source_text", "duration"]]
      .rename(columns={"source_utterance_id": "utt_id", "source_text": "text"})
      .drop_duplicates("utt_id").reset_index(drop=True))
ev["attack_eval_rank"] = np.arange(len(ev))
ev.to_csv(f"{WORK}/attack_eval_t1.csv", index=False)

# Publishing a new dataset version REPLACES the file set with whatever is in
# /kaggle/working. The hand-validated labels live in an attached input, not here,
# so without this copy they would vanish from the dataset on publish and the next
# run would fall back to "first round, unvalidated". Carry them through so the
# dataset is self-contained and re-running is idempotent.
if VAL:
    pd.read_csv(VAL).to_csv(f"{WORK}/t1_pairs_validated.csv", index=False)
    print(f"carried {len(judged)} validated labels into the output")
else:
    print("no validated labels to carry (first round)")

PROV.update(n_utterances=int(ev.utt_id.nunique()), n_rows=int(len(tg)),
            minutes=round((time.time()-T0)/60, 2))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
KU = os.environ.get("KAGGLE_USERNAME", "YOURUSERNAME")
json.dump({"title": "bn-attack-targets", "id": f"{KU}/bn-attack-targets",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)

print(json.dumps(PROV, indent=2))
print(f"\n{ev.utt_id.nunique()} utterances x 2 edit types, paired on the same audio.")
print("Publish as 'bn-attack-targets', then run NB-05 with EDIT_TYPE='T1' and 'T5'")
print("attaching it -- both will use this same set, so the comparison is paired.")
